In [1]:
!pip install -q huggingface_hub duckdb pandas

from getpass import getpass
from huggingface_hub import login, hf_hub_download
import duckdb, pandas as pd

hf_token = getpass("HF token: ")
login(token=hf_token)

def get_path(m):
    return hf_hub_download(
        repo_id="FlyRank/internship-warehouse", repo_type="dataset",
        filename=f"fact_content_daily_performance/month={m}/data_0.parquet"
    )

path_jan, path_feb, path_mar = get_path("2026-01"), get_path("2026-02"), get_path("2026-03")

con = duckdb.connect()
features_df = con.sql(f"""
    WITH filtered AS (
        SELECT * FROM read_parquet(['{path_jan}', '{path_feb}'])
        WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
    ),
    monthly AS (
        SELECT client_hash_id, content_hash_id, EXTRACT(month FROM report_date) AS m,
               AVG(gsc_clicks) AS avg_clicks
        FROM filtered GROUP BY 1,2,3
    )
    SELECT f.client_hash_id, f.content_hash_id,
        AVG(f.gsc_clicks) AS avg_daily_clicks,
        AVG(f.gsc_impressions) AS avg_daily_impressions,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS ai_sessions_share,
        MAX(CASE WHEN m.m = 2 THEN m.avg_clicks END) - MAX(CASE WHEN m.m = 1 THEN m.avg_clicks END) AS early_late_click_delta
    FROM filtered f JOIN monthly m ON f.client_hash_id = m.client_hash_id AND f.content_hash_id = m.content_hash_id
    GROUP BY 1,2
""").df()

label_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, AVG(gsc_clicks) AS label_window_avg_clicks
    FROM read_parquet('{path_mar}')
    WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
    GROUP BY 1,2
""").df()

data = features_df.merge(label_df, on=['client_hash_id','content_hash_id'], how='inner')
data = data.dropna(subset=['avg_daily_clicks','avg_daily_impressions','avg_position','ai_sessions_share','early_late_click_delta'])
data['is_declining'] = (data['label_window_avg_clicks'] < data['avg_daily_clicks'] * 0.8).astype(int)

print(data.shape)

HF token: ··········


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(21645, 9)


In [2]:
feat = data.copy()
feat['ctr'] = feat['avg_daily_clicks'] / feat['avg_daily_impressions'].replace(0, pd.NA)
feat = feat.dropna(subset=['ctr', 'avg_position'])

bins = [0, 3, 10, 20, 50, 100000]
labels = ['1-3', '4-10', '11-20', '21-50', '50+']
feat['position_bucket'] = pd.cut(feat['avg_position'], bins=bins, labels=labels)

signal1 = feat.groupby('position_bucket', observed=True).agg(avg_ctr=('ctr', 'mean'), n=('ctr', 'size'))
print(signal1)

                  avg_ctr     n
position_bucket                
1-3              0.002540   747
4-10             0.004346  8928
11-20            0.003758  7598
21-50            0.002773  3949
50+              0.002695   423


In [3]:
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_content.parquet"
)
dim_content = pd.read_parquet(dim_content_path, columns=['content_hash_id', 'last_optimized_date'])

merged = data.merge(dim_content, on='content_hash_id', how='left')
merged['last_optimized_date'] = pd.to_datetime(merged['last_optimized_date'])
decision_date = pd.Timestamp('2026-02-28')
merged['days_since_optimized'] = (decision_date - merged['last_optimized_date']).dt.days

bins2 = [-1, 30, 90, 180, 100000]
labels2 = ['<30d', '30-90d', '90-180d', '180d+']
merged['staleness_bucket'] = pd.cut(merged['days_since_optimized'], bins=bins2, labels=labels2)

signal2 = merged.groupby('staleness_bucket', observed=True).agg(decline_rate=('is_declining', 'mean'), n=('is_declining', 'size'))
print(signal2)

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Empty DataFrame
Columns: [decline_rate, n]
Index: []


In [5]:
expected_ctr_map = signal1['avg_ctr'].to_dict()
feat['expected_ctr'] = feat['position_bucket'].map(expected_ctr_map).astype(float)
feat['ctr_gap'] = feat['expected_ctr'] - feat['ctr']
feat['score'] = feat['ctr_gap']
feat['reason_code'] = 'CTR_BELOW_EXPECTED_FOR_POSITION'

threshold = feat['ctr_gap'].quantile(0.75)
feat['action'] = feat['ctr_gap'].apply(lambda g: 'CTR_FIX' if g >= threshold else 'MONITOR')

queue = feat[['client_hash_id','content_hash_id','avg_position','ctr','expected_ctr','ctr_gap','score','reason_code','action']] \
    .sort_values('score', ascending=False).reset_index(drop=True)

print(threshold)
queue.head(10)

0.002994289590582724


,client_hash_id,content_hash_id,avg_position,ctr,expected_ctr,ctr_gap,score,reason_code,action
0,client_e547b89c05043229,content_1de96bb8ad460b05,9.856137,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
1,client_3197e6291363b4db,content_97a856421e2fd750,6.400000,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
2,client_2094c6eb080311d5,content_df1b2ed6ada0d3e4,5.500000,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
3,client_65de48885f4ef01b,content_c4354cf6d438d9e2,8.935847,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
4,client_e547b89c05043229,content_0e8c6d057a221a2f,5.721411,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
5,client_3197e6291363b4db,content_30e625001db3bc05,8.696586,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
6,client_e547b89c05043229,content_a8cec44936afb836,4.597379,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
7,client_e547b89c05043229,content_62ade1fa64c7e058,8.639861,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
8,client_e547b89c05043229,content_c80d368b1bc74f35,6.204923,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
9,client_e547b89c05043229,content_0dc2023bd029be11,5.568687,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX


In [6]:
import os, json

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

metrics = {
    "signal1_ctr_vs_position": signal1.reset_index().to_dict(orient='records'),
    "signal2_staleness_vs_decline": signal2.reset_index().to_dict(orient='records'),
    "rule_threshold_ctr_gap": float(threshold),
    "n_flagged_ctr_fix": int((feat['action'] == 'CTR_FIX').sum()),
    "n_total": int(len(feat))
}
with open('work/outputs/w04_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print("done")

done


In [7]:
queue.head(10)

,client_hash_id,content_hash_id,avg_position,ctr,expected_ctr,ctr_gap,score,reason_code,action
0,client_e547b89c05043229,content_1de96bb8ad460b05,9.856137,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
1,client_3197e6291363b4db,content_97a856421e2fd750,6.400000,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
2,client_2094c6eb080311d5,content_df1b2ed6ada0d3e4,5.500000,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
3,client_65de48885f4ef01b,content_c4354cf6d438d9e2,8.935847,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
4,client_e547b89c05043229,content_0e8c6d057a221a2f,5.721411,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
5,client_3197e6291363b4db,content_30e625001db3bc05,8.696586,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
6,client_e547b89c05043229,content_a8cec44936afb836,4.597379,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
7,client_e547b89c05043229,content_62ade1fa64c7e058,8.639861,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
8,client_e547b89c05043229,content_c80d368b1bc74f35,6.204923,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX
9,client_e547b89c05043229,content_0dc2023bd029be11,5.568687,0.0,0.004346,0.004346,0.004346,CTR_BELOW_EXPECTED_FOR_POSITION,CTR_FIX


In [8]:
import os
print(os.getcwd())
print(os.path.exists('work/outputs/baseline_action_score.csv'))
print(os.path.exists('work/outputs/w04_metrics.json'))
print(os.listdir('work/outputs') if os.path.exists('work/outputs') else "folder missing")

/content
True
True
['baseline_action_score.csv', 'w04_metrics.json']
